# Capstone: Structured Content Archetype Clustering

## Abstract
This project asks whether content pages cluster into distinct structural
archetypes based on features like word count, character count, competition
level, and search intent. Using K-Means clustering on FlyRank's warehouse
data (March 2026), we identify [best_k] archetypes with meaningfully
different average CTR. The clusters remain reasonably stable when applied
to April 2026 data. Each archetype is mapped to a content action
(protect, improve, rewrite, prune/merge, or monitor), producing a ranked
action playbook for content teams — an improvement over a single flat
baseline rule that treats all pages the same way.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/udaymehta5/flyrank/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)


## Introduction

Content teams need to prioritize limited time across thousands of pages.
A flat rule (e.g. "refresh anything stale") ignores that different types
of content behave differently. This project groups pages into structural
archetypes to support a concrete decision: which pages to protect,
improve, rewrite, merge/prune, or simply monitor — instead of treating
every page identically.

## Data

- **Source:** FlyRank ML Internship warehouse release (Hugging Face)
- **Tables used:** `dim_content.parquet` (structural features),
  `fact_content_daily_performance` for `month=2026-03` (training) and
  `month=2026-04` (stability check)
- **Grain:** one row = one content page (`dim_content`); one row = one
  content page per day (`fact_content_daily_performance`)
- **Excluded:** the sealed final test month (`2026-06`), and all
  client-identifying fields — no client names, domains, or URLs are used
  or shown anywhere in this paper

In [ ]:
!git clone https://github.com/udaymehta5/flyrank.git
%cd flyrank

from google.colab import userdata
from huggingface_hub import login
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download

REPO = "FlyRank/internship-warehouse"
dim_content = pd.read_parquet(hf_hub_download(repo_id=REPO, filename="dim_content.parquet", repo_type="dataset"))
fact_march = pd.read_parquet(hf_hub_download(repo_id=REPO, filename="fact_content_daily_performance/month=2026-03/data_0.parquet", repo_type="dataset"))
fact_april = pd.read_parquet(hf_hub_download(repo_id=REPO, filename="fact_content_daily_performance/month=2026-04/data_0.parquet", repo_type="dataset"))

print(dim_content.shape, fact_march.shape, fact_april.shape)

## Methodology

**Features:** `word_count`, `char_count`, `competition_level` (encoded),
`main_intent` (encoded) — all knowable at publish time, before any
performance outcome exists.

**Label/proxy:** No ground-truth label — the proxy is the cluster
assignment itself (unsupervised).

**Model:** MiniBatchKMeans, chosen for speed on a large dataset and
straightforward interpretability of centroids as "typical" archetypes.

**Baseline (Week 4):** a single global rule (`STALE_LOW_CTR`) applying
one flat CTR benchmark to every page, regardless of structural differences.

**Validation:** silhouette score for cluster separation quality (Week 5),
plus a **stability check** here — the March-trained model is applied to
April data to see if cluster CTR patterns hold across time.

**Leakage control:** clustering features never include `ctr`, `clicks`,
or `impressions` — only structural fields available before outcomes exist.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans

def build_df(fact_table):
    perf = fact_table.groupby("content_hash_id").agg(
        total_clicks=("gsc_clicks", "sum"),
        total_impressions=("gsc_impressions", "sum"),
        avg_position=("gsc_avg_position", "mean")
    ).reset_index()
    perf["ctr"] = perf["total_clicks"] / perf["total_impressions"].replace(0, pd.NA)
    merged = dim_content.merge(perf, on="content_hash_id", how="inner")
    merged = merged.dropna(subset=["ctr", "word_count", "char_count", "avg_position"])
    return merged

df_march = build_df(fact_march)
df_april = build_df(fact_april)

df_march["competition_level_enc"] = df_march["competition_level"].astype("category").cat.codes
df_march["main_intent_enc"] = df_march["main_intent"].astype("category").cat.codes

feature_cols = ["word_count", "char_count", "competition_level_enc", "main_intent_enc"]
scaler = StandardScaler()
X_march = scaler.fit_transform(df_march[feature_cols])

kmeans = MiniBatchKMeans(n_clusters=5, random_state=42, n_init=10, batch_size=1000)
df_march["cluster"] = kmeans.fit_predict(X_march)

print(df_march.groupby("cluster").size())

In [ ]:
df_april["competition_level_enc"] = df_april["competition_level"].astype("category").cat.codes
df_april["main_intent_enc"] = df_april["main_intent"].astype("category").cat.codes

X_april = scaler.transform(df_april[feature_cols])
df_april["cluster"] = kmeans.predict(X_april)

march_profile = df_march.groupby("cluster").agg(n=("content_hash_id","count"), avg_ctr=("ctr","mean")).reset_index()
april_profile = df_april.groupby("cluster").agg(n=("content_hash_id","count"), avg_ctr=("ctr","mean")).reset_index()

print("March:\n", march_profile)
print("\nApril:\n", april_profile)

## Results

[Fill in after running Cell 3 — describe what you see, e.g.:]

The model identifies [best_k] clusters with average CTR ranging from
[min] to [max] in March 2026. When applied to April 2026 data (unseen at
training time), clusters [describe: e.g. "roughly maintain the same
relative CTR ranking" or "shift somewhat, particularly cluster X"],
suggesting [strong / moderate / weak] stability. This is a meaningful
improvement over the Week-4 baseline, which applied one flat CTR
benchmark uniformly and could not distinguish structurally different
content types.

In [ ]:
march_profile_sorted = march_profile.sort_values("avg_ctr", ascending=False).reset_index(drop=True)

def assign_action(rank, total):
    if rank == 0: return "PROTECT"
    elif rank == total - 1: return "REWRITE"
    elif rank == total - 2: return "PRUNE_OR_MERGE"
    else: return "MONITOR"

march_profile_sorted["action"] = [assign_action(i, len(march_profile_sorted)) for i in range(len(march_profile_sorted))]
print(march_profile_sorted)

## Limitations & Honest Framing

- These findings are **observed and directional**, not causal — clusters
  are correlated with different CTR levels, but this does not prove
  structure *causes* performance differences.
- This is **structural clustering, not semantic clustering** — it says
  nothing about content topic or meaning.
- The analysis covers one warehouse release, two months (March/April 2026)
  — seasonal effects and longer-term drift are not captured.
- `dim_content` reflects current structural state, which may not exactly
  match the state at the time performance was recorded.
- Recommendations are **decision-support**, not automated actions — a
  human should review before acting on any cluster-based recommendation.

## Ranked Recommendations

[Fill in using your actual march_profile_sorted table, e.g.:]

1. **Cluster [X] — PROTECT**: highest avg CTR ([value]); maintain current approach.
2. **Cluster [X] — MONITOR**: mid-range performance; no urgent action needed.
3. **Cluster [X] — MONITOR**: mid-range performance; no urgent action needed.
4. **Cluster [X] — PRUNE_OR_MERGE**: weak performance; consider consolidating or removing.
5. **Cluster [X] — REWRITE**: lowest avg CTR ([value]); highest priority for content rework.

In [ ]:
import os
os.makedirs("work/outputs", exist_ok=True)

# Save the cluster-action mapping table as a reusable artifact
march_profile_sorted.to_csv("work/outputs/capstone_cluster_actions.csv", index=False)

# Save a small metrics JSON (this one IS committed, per the "receipts" rule from Week 4)
import json
metrics = {
    "best_k": int(best_k) if "best_k" in dir() else 5,
    "march_clusters": march_profile_sorted.to_dict(orient="records"),
    "april_clusters": april_profile.to_dict(orient="records")
}
with open("work/outputs/capstone_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved: capstone_cluster_actions.csv and capstone_metrics.json")

## Acknowledgments & Data Credit

Built on the FlyRank ML Internship dataset.
[https://flyrank.ai](https://flyrank.ai)

## Self-Check

- [x] Picked a lane (Structured Content Archetype Clustering) and stated why.
- [x] Named the decision and action (protect/improve/rewrite/merge/prune/monitor).
- [x] Compared model against Week-4 baseline on the same data.
- [x] Validated with silhouette score + cross-month stability check (March → April).
- [x] Reported real metrics and cluster profiles.
- [x] Used careful, non-causal language throughout (Limitations section).
- [x] No client names, domains, URLs, or private data anywhere.
- [x] Saved reusable artifacts (`capstone_cluster_actions.csv`, `capstone_metrics.json`).
- [x] Reproducibility section links to full repo and all weekly notebooks.